# Kim1

A single stationary objective on $[-16,16]^2$: a sinusoidal ripple on an
asymmetric quadratic bowl. Runtime is a few minutes.

In [ ]:
import sys, time, pickle
sys.path.insert(0, '..')          # drop this line if the notebook is at the root

import numpy as np
from joblib import Parallel, delayed

from ibo import (sigma_estimator, get_automatic_kernel_args,
                 evaluate_single_bo, kim1,
                 get_benchmarks, shift_benchmarks,
                 sample_gp_function_, generate_centers,
                 plot_grouped_comparison, plot_mean_ratio_across_functions,
                 report_final_gap, paired_comparison)

In [ ]:
bounds_kim1 = np.array([[-16, 16], [-16, 16]])
d = bounds_kim1.shape[0]

tree_depth = 3
n_init     = 40 * d
n_iter     = 200
n_seeds    = 30
min_leaf   = 16
tau        = 1.5          # splitting threshold, shared across all suites
noise_var  = 1e-8

The length-scale is fitted once on an independent Sobol design of
$n_{init}+n_{iter}$ points and then held fixed.

In [ ]:
l_kim1 = get_automatic_kernel_args(
    {'Kim1': lambda X: np.asarray(kim1(np.atleast_2d(X))).ravel()},
    d=d, n_points=n_init + n_iter, bounds=(-16.0, 16.0))
kernel_args = l_kim1['Kim1']
kernel_args

In [ ]:
configs = [
    # --- EI Group ---
    ("Classic BO - EI", 'fixed', 1,  False, 'ei', {'xi': 0.0}),
    ("IBO_1 - EI",      'fixed', 1,  True,  'ei', {'xi': 0.0}),
    ("IBO_4 - EI",      'fixed', 2,  True,  'ei', {'xi': 0.0}),
    ("IBO_P - EI",      'tree',  2,  True,  'ei', {'xi': 0.0}),

    # --- PI Group ---
    ("Classic BO - PI", 'fixed', 1,  False, 'pi', {'xi': 0.0}),
    ("IBO_1 - PI",      'fixed', 1,  True,  'pi', {'xi': 0.0}),
    ("IBO_4 - PI",      'fixed', 2,  True,  'pi', {'xi': 0.0}),
    ("IBO_P - PI",      'tree',  2,  True,  'pi', {'xi': 0.0}),

    # --- LCB Group ---
    ("Classic BO - LCB", 'fixed', 1, False, 'lcb', {'beta': 1.96}),
    ("IBO_1 - LCB",      'fixed', 1, True,  'lcb', {'beta': 1.96}),
    ("IBO_4 - LCB",      'fixed', 2, True,  'lcb', {'beta': 1.96}),
    ("IBO_P - LCB",      'tree',  2, True,  'lcb', {'beta': 1.96}),

    # --- TS Group ---
    ("Classic BO - TS", 'fixed', 1, False, 'ts', {}),
    ("IBO_1 - TS",      'fixed', 1, True,  'ts', {}),
    ("IBO_4 - TS",      'fixed', 2, True,  'ts', {}),
    ("IBO_P - TS",      'tree',  2, True,  'ts', {}),
]

In [ ]:
start = time.perf_counter()
print(f"--- Running {n_seeds} seeds in parallel for Kim1 (tau = {tau}) ---")

results_kim1 = Parallel(n_jobs=-1)(
    delayed(evaluate_single_bo)(
        seed, kim1, bounds_kim1, configs, n_init, d, n_iter,
        kernel_args, noise_var, tree_depth, sigma_estimator,
        tau=tau, n_candidates=256, min_leaf=min_leaf, return_info=True,
    ) for seed in range(n_seeds)
)

histories_kim1 = {c[0]: np.zeros((n_seeds, n_iter + 1)) for c in configs}
for seed, seed_histories, infos in results_kim1:
    for label, history in seed_histories.items():
        histories_kim1[label][seed, :] = history

leaves = [info["n_leaves_hist"].mean()
          for _, _, infos in results_kim1
          for label, info in infos.items() if label.startswith("IBO_P")]
print(f"IBO_P mean leaves {np.mean(leaves):.2f}, "
      f"split in {100*np.mean([l > 1 for l in leaves]):.0f}% of runs")
print(f"Execution time: {time.perf_counter() - start:.1f} s")

In [ ]:
plot_grouped_comparison(histories_kim1, n_seeds=n_seeds,
                        save_path='kim1_grouped.pdf')

In [ ]:
tag = f"tau{tau}_leaf{min_leaf}_init{n_init}_iter{n_iter}"
with open(f'../results/kim1_{tag}.pkl', 'wb') as fh:
    pickle.dump({'histories': histories_kim1,
                 'leaves': float(np.mean(leaves)),
                 'kernel_args': kernel_args,
                 'params': {'tau': tau, 'min_leaf': min_leaf, 'n_init': n_init,
                            'n_iter': n_iter, 'n_seeds': n_seeds,
                            'tree_depth': tree_depth}}, fh)
print(f'saved kim1_{tag}.pkl')